In [6]:

# 영어 직무 설명 텍스트 기반 군집 분석
# + 군집별 성향(trait) 유사도 계산

import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity


# 1) 위키피디아 기반 직무 설명 데이터 로딩
# 각 행은 하나의 직무 설명 텍스트로 구성됨 약 50개
df = pd.read_csv("wikipedia_software_jobs_large.csv")

texts_raw = df["full_text"].fillna("")




# 2)특수문자 제거 및 소문자 변환하기 위해 전처리
def clean_text(text):
    text = re.sub(r"[^0-9A-Za-z\s]", " ", str(text))
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

texts = texts_raw.apply(clean_text)



# 3) TF-IDF 벡터화
# 직무 설명 텍스트를 수치 벡터로 변환 -> KMeans 알고리즘 적용 위해
vectorizer_roles = TfidfVectorizer(
    max_features=4000,
    stop_words="english"
)
X_roles = vectorizer_roles.fit_transform(texts)



# 4) KMeans 군집 분석
# 직무 특성에 따라 군집화
k = 6
kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=20
)

df["cluster"] = kmeans.fit_predict(X_roles)



# 5) 군집별 대표 키워드 추출
terms = vectorizer_roles.get_feature_names_out()
order = kmeans.cluster_centers_.argsort()[:, ::-1]

cluster_top_terms = {}
for cid in range(k):
    cluster_top_terms[cid] = [
        terms[idx] for idx in order[cid, :25]
    ]

# 6) 군집 설명 문서 구성
# 군집 키워드 + 보조 힌트를 결합하여 문서화
cluster_role_hint = {
    0: "system programming os network architecture embedded security",
    1: "testing qa devops security automation performance",
    2: "data ai machine learning deep learning statistics",
    3: "web app backend frontend mobile cloud service",
    4: "product management ux qa consulting documentation",
    5: "game graphics engine realtime firmware vr ar"
}

cluster_sentences = {}
for cid, words in cluster_top_terms.items():
    cluster_sentences[cid] = " ".join(words) + " " + cluster_role_hint[cid]


# 7) 성향(trait) 기준 문서 정의
# 각 성향을 대표하는 기술 키워드 집합
trait_texts = {
    "logic": "...",
    "creative": "...",
    "analysis": "...",
    "stability": "..."
}

trait_names = list(trait_texts.keys())


# 8) 성향 문서 + 군집 문서 TF-IDF 변환
all_texts = list(trait_texts.values()) + list(cluster_sentences.values())

vectorizer_all = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)
tfidf_all = vectorizer_all.fit_transform(all_texts)

trait_vectors = tfidf_all[:4]
cluster_vectors = tfidf_all[4:]


# 9) 군집-성향 간 코사인 유사도 계산
cluster_trait_scores = {}
for cid in range(k):
    sims = cosine_similarity(cluster_vectors[cid], trait_vectors)[0]
    cluster_trait_scores[cid] = {
        trait_names[i]: float(sims[i])
        for i in range(len(trait_names))
    }



# 10) 군집별 대표 성향 선택
cluster_main_trait = {
    cid: max(scores, key=scores.get)
    for cid, scores in cluster_trait_scores.items()
}


# 11) 성향 → 트랙 이름 매핑
trait_to_track = {
    "logic": "시스템·백엔드 구조 설계 트랙",
    "creative": "프론트엔드·UX 중심 서비스 개발 트랙",
    "analysis": "AI·데이터 분석 및 지능형 시스템 트랙",
    "stability": "클라우드·DevOps·서비스 운영 트랙"
}

cluster_track_label = {
    cid: trait_to_track[trait]
    for cid, trait in cluster_main_trait.items()
}


In [7]:
import numpy as np

# ✅ 성향 벡터 순서 정의 (필수)
TRAIT_ORDER = ["logic", "creative", "analysis", "stability"]

cluster_trait_vectors = {}

for cid, scores in cluster_trait_scores.items():
    vec = np.array(
        [scores[t] for t in TRAIT_ORDER],
        dtype=float
    )
    cluster_trait_vectors[cid] = vec


In [8]:
cluster_trait_vectors

{0: array([0., 0., 0., 0.]),
 1: array([0., 0., 0., 0.]),
 2: array([0., 0., 0., 0.]),
 3: array([0., 0., 0., 0.]),
 4: array([0., 0., 0., 0.]),
 5: array([0., 0., 0., 0.])}

In [6]:
for cid in range(6):
    print(f"\n=== Cluster {cid} 직무 목록 ===")
    print(df[df["cluster"] == cid]["input_job"].tolist())



=== Cluster 0 직무 목록 ===
['Software engineer', 'Systems programmer', 'Embedded systems engineer', 'Computer vision engineer', 'Recommendation systems engineer', 'Site reliability engineer', 'Network engineer', 'Systems engineer', 'Information security engineer', 'Digital forensics specialist', 'UI designer', 'Computer science researcher', 'Robotics engineer', 'HCI researcher', 'Autonomous vehicle engineer', 'Solutions architect', 'Open source developer']

=== Cluster 1 직무 목록 ===
['DevOps engineer', 'Penetration tester', 'Software tester', 'Test automation engineer', 'Performance test engineer']

=== Cluster 2 직무 목록 ===
['Data scientist', 'Data engineer', 'Machine learning engineer', 'Deep learning researcher', 'Natural language processing engineer', 'AI engineer', 'Startup founder']

=== Cluster 3 직무 목록 ===
['Frontend developer', 'Backend developer', 'Full-stack developer', 'Web developer', 'Mobile app developer', 'Android developer', 'iOS developer', 'Cloud engineer']

=== Cluster 4 직

In [23]:
#FastAPI용 pkl 저장 
import pickle
import os

PROJECT_ROOT = r"C:\Python\Python311\Scripts\데크_기말고사"
BACKEND_DIR = os.path.join(PROJECT_ROOT, "backend")

os.makedirs(BACKEND_DIR, exist_ok=True)

# 1) 클러스터별 trait 점수
with open(os.path.join(BACKEND_DIR, "cluster_trait_scores.pkl"), "wb") as f:
    pickle.dump(cluster_trait_scores, f)

# 2) 클러스터 → 트랙 이름 매핑
with open(os.path.join(BACKEND_DIR, "cluster_track_label.pkl"), "wb") as f:
    pickle.dump(cluster_track_label, f)

print("pkl 생성 완료")

✅ FastAPI용 pkl 생성 완료
📁 저장 위치: C:\Python\Python311\Scripts\데크_기말고사\backend


In [4]:
print(type(cluster_trait_scores))
print(cluster_track_label)


<class 'dict'>
{0: '시스템·백엔드 구조 설계 트랙', 1: '클라우드·DevOps·서비스 운영 트랙', 2: 'AI·데이터 분석 및 지능형 시스템 트랙', 3: '시스템·백엔드 구조 설계 트랙', 4: '프론트엔드·UX 중심 서비스 개발 트랙', 5: '시스템·백엔드 구조 설계 트랙'}
